# SQL in PySpark

## What This Notebook Demonstrates
This notebook teaches you how to use SQL queries within PySpark notebooks. You'll learn multiple ways to execute SQL, create temporary views from DataFrames, and understand when to use each approach.

---

## Key Concepts Explained

### **1. Two Ways to Run SQL Queries**

#### **Method 1: Using `spark.sql()` in Python**
```python
df = spark.sql("SELECT * FROM table_name WHERE condition")
display(df)
```

**Concepts:**
* **`spark.sql()`** - Executes SQL query and returns a DataFrame
* **Returns a DataFrame object** - You can store it in a variable and apply further transformations
* **Best for:** Mixing SQL with Python transformations, reusing query results
* **Note:** The `.display()` in the assignment returns None, so you need `display(df)` separately

#### **Method 2: Using SQL Cells (with `%sql` magic)** ⚡ *Simpler for pure SQL*
```sql
SELECT * FROM table_name WHERE condition
```

**Concepts:**
* **SQL cell** - Entire cell is treated as SQL (indicated by SQL language selector)
* **Automatically displays results** - No need for display() function
* **Best for:** Pure SQL queries, quick data exploration
* **Results stored in** `_sqldf` variable (accessible in Python cells)

**When to use which?**
* Use **spark.sql()** when you need to chain Python operations
* Use **SQL cells** for standalone queries and reports

---

### **2. Creating DataFrames**

```python
data = [("2017-01-01", 32.0, "Rain"), ("2017-01-02", 28.0, "Snow")]
schema = "day string, temperature double, event string"
df = spark.createDataFrame(data, schema)
```

**Concepts:**
* **`spark.createDataFrame()`** - Creates DataFrame from Python data structures
* **Data parameter** - List of tuples (each tuple = one row)
* **Schema definition (string format):**
  * Format: `"column_name data_type, column_name data_type, ..."`
  * Common types: `string`, `int`, `double`, `date`, `timestamp`, `boolean`
  * Alternative: Can also use StructType (like in read_write_csv notebook)

---

### **3. Data Transformation Functions**

```python
from pyspark.sql import functions as F
df = df.withColumn("day", F.to_date("day", "yyyy-MM-dd"))
```

**Concepts:**
* **`F.to_date(column, format)`** - Converts string to date type
  * First parameter: Column name to convert
  * Second parameter: Date format pattern
    * `yyyy` = 4-digit year
    * `MM` = 2-digit month
    * `dd` = 2-digit day
* **`withColumn(name, expression)`** - Adds new column or replaces existing one
  * If column name exists: replaces it
  * If column name is new: adds it

**Why convert to date type?**
* ✅ Enables date arithmetic (add days, calculate differences)
* ✅ Proper sorting and filtering by date
* ✅ Access to date functions (YEAR(), MONTH(), DAY())

---

### **4. Temporary Views - Bridge Between DataFrames and SQL**

#### **Session-Scoped Temp View**
```python
df.createOrReplaceTempView("weather")
```

**Concepts:**
* **Creates a temporary SQL table** from a DataFrame
* **Session-scoped** - Only available in current SparkSession
* **Disappears when:** Notebook detaches or session ends
* **Query with:** `SELECT * FROM weather`
* **Use case:** Share data between Python and SQL cells in same notebook

#### **Global Temp View**
```python
df.createGlobalTempView("global_weather")
```

**Concepts:**
* **Global scope** - Available across different SparkSessions
* **Stored in special database:** `global_temp`
* **Query with:** `SELECT * FROM global_temp.global_weather`
* **Use case:** Share data between different notebooks in same cluster
* **Still temporary** - Disappears when cluster/application terminates

**Comparison:**

| Feature | Temp View | Global Temp View |
|---------|-----------|------------------|
| Scope | Single session | All sessions in cluster |
| Query syntax | `FROM view_name` | `FROM global_temp.view_name` |
| Lifetime | Session ends | Cluster terminates |
| Best for | Within notebook | Between notebooks |

---

### **5. SQL Aggregation Functions**

```sql
SELECT event, ROUND(AVG(temperature), 1) AS avg_temp
FROM weather
GROUP BY event
ORDER BY avg_temp DESC
```

**Concepts:**
* **`AVG(column)`** - Calculates average of numeric column
* **`ROUND(value, decimals)`** - Rounds number to specified decimal places
  * `ROUND(32.456, 1)` → 32.5
  * `ROUND(32.456, 0)` → 32
* **`GROUP BY column`** - Groups rows by unique values in column
  * Collapses multiple rows into summary rows
  * Required when mixing regular columns with aggregate functions
* **`ORDER BY column DESC`** - Sorts results
  * `DESC` = descending (highest first)
  * `ASC` = ascending (lowest first, default)
* **`AS alias`** - Renames column in output

**How aggregation works:**
1. `GROUP BY event` groups all rows with same event together
2. `AVG(temperature)` calculates average within each group
3. `ROUND(..., 1)` rounds to 1 decimal place
4. Results sorted by temperature (highest to lowest)

---

## Handling NULL Values

Notice the sample data includes `None` values (Python) which become `NULL` in Spark:
* **`AVG()` automatically ignores NULL values** - calculates average of non-null values only
* **`COUNT(column)` skips NULLs** - counts only non-null values
* **`COUNT(*)` includes NULLs** - counts all rows

---

## Data Flow Pattern

```
Python Data → DataFrame → Temp View → SQL Query → Results
     ↓            ↓           ↓            ↓
  (tuples)   (structured)  (table)   (aggregated)
```

---

## Best Practices Demonstrated

1. **Convert string dates to date type** for proper date operations
2. **Use temp views** to query DataFrames with SQL
3. **Choose the right view scope** (session vs global)
4. **Use SQL for aggregations** - often clearer than PySpark syntax
5. **Round numeric results** for readability
6. **Use meaningful aliases** (AS) for calculated columns
7. **Import functions once** at the top (`from pyspark.sql import functions as F`)

---

## Quick Reference

| Task | Python/PySpark | SQL |
|------|----------------|-----|
| Query table | `spark.sql("SELECT...")` | SQL cell with query |
| Create view | `df.createOrReplaceTempView("name")` | N/A |
| Average | `F.avg("column")` | `AVG(column)` |
| Round | `F.round("column", 2)` | `ROUND(column, 2)` |
| Group by | `df.groupBy("column")` | `GROUP BY column` |
| Sort | `df.orderBy("column")` | `ORDER BY column` |

## SQL Using Python

In [0]:
df_marvel = spark.sql("select * from workspace.default.movies where studio = 'Marvel Studios'").display()
display(df_marvel)

title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language
Doctor Strange in the Multiverse of Madness,Hollywood,2022,7,Marvel Studios,200.0,954.8,Millions,USD,English
Thor: The Dark World,Hollywood,2013,6.8,Marvel Studios,165.0,644.8,Millions,USD,English
Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,180.0,854.0,Millions,USD,English
Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,250.0,670.0,Millions,USD,English
Avengers: Endgame,Hollywood,2019,8.4,Marvel Studios,400.0,2798.0,Millions,USD,English
Avengers: Infinity War,Hollywood,2018,8.4,Marvel Studios,400.0,2048.0,Millions,USD,English
Captain America: The First Avenger,Hollywood,2011,6.9,Marvel Studios,216.7,370.6,Millions,USD,English
Captain America: The Winter Soldier,Hollywood,2014,7.8,Marvel Studios,177.0,714.4,Millions,USD,English


## SQL Using %sql

In [0]:
%sql
SELECT * FROM workspace.default.movies WHERE studio = 'Marvel Studios'

title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language
Doctor Strange in the Multiverse of Madness,Hollywood,2022,7,Marvel Studios,200.0,954.8,Millions,USD,English
Thor: The Dark World,Hollywood,2013,6.8,Marvel Studios,165.0,644.8,Millions,USD,English
Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,180.0,854.0,Millions,USD,English
Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,250.0,670.0,Millions,USD,English
Avengers: Endgame,Hollywood,2019,8.4,Marvel Studios,400.0,2798.0,Millions,USD,English
Avengers: Infinity War,Hollywood,2018,8.4,Marvel Studios,400.0,2048.0,Millions,USD,English
Captain America: The First Avenger,Hollywood,2011,6.9,Marvel Studios,216.7,370.6,Millions,USD,English
Captain America: The Winter Soldier,Hollywood,2014,7.8,Marvel Studios,177.0,714.4,Millions,USD,English


In [0]:
from pyspark.sql import functions as F, types as T

data = [
    ("2017-01-01", 32.0, 6.0, "Rain"),
    ("2017-01-04", None, 9.0, "Sunny"),
    ("2017-01-05", 28.0, None, "Snow"),
    ("2017-01-06", None, 7.0, None),
    ("2017-01-07", 32.0, None, "Rain"),
    ("2017-01-08", None, None, "Sunny"),
    ("2017-01-09", None, None, None),
    ("2017-01-10", 34.1, 8.1, "Cloudy"),
    ("2017-01-11", 40.0, 12.0, "Sunny")
]

schema = "day string, temperature double, windspeed double, event string"
df = spark.createDataFrame(data, schema)
df = df.withColumn("day", F.to_date("day", "yyyy-MM-dd"))

display(df)

day,temperature,windspeed,event
2017-01-01,32.0,6.0,Rain
2017-01-04,null,9.0,Sunny
2017-01-05,28.0,null,Snow
2017-01-06,null,7.0,null
2017-01-07,32.0,null,Rain
2017-01-08,null,null,Sunny
2017-01-09,null,null,null
2017-01-10,34.1,8.1,Cloudy
2017-01-11,40.0,12.0,Sunny


In [0]:
df.createOrReplaceTempView("weather") # session scoped
df.createGlobalTempView("global_weather")

In [0]:
%sql
SELECT event, ROUND(AVG(temperature),1) AS avg_temp
FROM weather
GROUP BY event
ORDER BY avg_temp DESC

event,avg_temp
Sunny,40.0
Cloudy,34.1
Rain,32.0
Snow,28.0
null,null
